# ClimateGuard AI v3.0 - Feature Selection

This notebook covers feature selection techniques to identify the most important features for ML models. 

## 1. Import Libraries 

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from src.feature_engineering.feature_selection import FeatureSelector

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load EDA Data 

In [2]:
# Load data from EDA step
# Load data from EDA step
eda_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed', '04_eda_data.csv')
if not os.path.exists(eda_path):
    raise FileNotFoundError(f"EDA data file not found. Expected path: {eda_path}")
df = pd.read_csv(eda_path)

print(f"Data loaded: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Data loaded: {df.shape}")
print(f"Columns: {len(df.columns)}")

Data loaded: (119398, 87)
Columns: 87
Data loaded: (119398, 87)
Columns: 87


## 3. Prepare Features and Target 

In [3]:
# Select numerical features for selection
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X = df[numerical_cols].fillna(0)

# Create a target for supervised feature selection (e.g., rainfall)
if 'precipitation' in df.columns:
    y = (df['precipitation'] > 0).astype(int)  # Binary: rain or no rain
else:
    y = None

print(f"Features: {len(X.columns)}")
print(f"Target: {'Rainfall (binary)' if y is not None else 'None'}")

Features: 61
Target: Rainfall (binary)


## 4. Initialize Feature Selector 

In [4]:
selector = FeatureSelector(X, y)
print("Feature selector initialized")

Feature selector initialized


## 5. Variance Threshold Selection 

In [5]:
variance_features = selector.variance_threshold_selection(threshold=0.01)
print(f"Variance threshold selected {len(variance_features)} features")
print(f"Removed: {len(X.columns) - len(variance_features)} low-variance features")

INFO:src.feature_engineering.feature_selection:Variance threshold selected 54 features


Variance threshold selected 54 features
Removed: 7 low-variance features


## 6. Correlation-Based Selection 

In [6]:
correlation_features = selector.correlation_selection(threshold=0.95)
print(f"Correlation selection kept {len(correlation_features)} features")
print(f"Removed: {len(X.columns) - len(correlation_features)} highly correlated features")

INFO:src.feature_engineering.feature_selection:Correlation selection removed 15 features


Correlation selection kept 46 features
Removed: 15 highly correlated features


## 7. Univariate Selection (F-test)

In [7]:
if y is not None:
    univariate_features = selector.univariate_selection(k=20, score_func='f_classif')
    print(f"Univariate selection selected {len(univariate_features)} features")
    print("Top features by F-score:")
    report = selector.get_selection_report()
    if 'univariate' in report and 'feature_scores' in report['univariate']:
        sorted_features = sorted(report['univariate']['feature_scores'].items(), 
                               key=lambda x: x[1], reverse=True)
        for feat, score in sorted_features[:10]:
            print(f"  {feat}: {score:.4f}")
else:
    print("Target not available for univariate selection")

INFO:src.feature_engineering.feature_selection:Univariate selection selected 20 features


Univariate selection selected 20 features
Top features by F-score:
  rain_flag: inf
  cloud_cover: 99002.3400
  humidity: 20973.6386
  precipitation: 18599.3304
  rainfall_intensity: 18599.3304
  precip_in: 17745.8350
  rain_rolling_7d: 15082.4098
  dew_point: 13416.3294
  air_quality_gb-defra-index: 10045.1604
  air_quality_us-epa-index: 9579.0106


## 8. Random Forest Importance

In [8]:
if y is not None:
    rf_features = selector.random_forest_importance(n_features=20)
    print(f"Random Forest importance selected {len(rf_features)} features")
    print("Top features by importance:")
    report = selector.get_selection_report()
    if 'rf_importance' in report and 'feature_importances' in report['rf_importance']:
        sorted_features = sorted(report['rf_importance']['feature_importances'].items(), 
                               key=lambda x: x[1], reverse=True)
        for feat, imp in sorted_features[:10]:
            print(f"  {feat}: {imp:.4f}")
else:
    print("Target not available for RF importance")

INFO:src.feature_engineering.feature_selection:Random Forest importance selected 20 features


Random Forest importance selected 20 features
Top features by importance:
  rainfall_intensity: 0.2539
  rain_flag: 0.2490
  precipitation: 0.2330
  precip_in: 0.0903
  cloud_cover: 0.0576
  rain_rolling_7d: 0.0439
  humidity: 0.0098
  dew_point: 0.0081
  pm10: 0.0071
  feels_like_diff: 0.0070


## 9. Recursive Feature Elimination

In [9]:
if y is not None:
    rfe_features = selector.recursive_feature_elimination(n_features=15)
    print(f"RFE selected {len(rfe_features)} features")
    print("Selected features:")
    for feat in rfe_features:
        print(f"  {feat}")
else:
    print("Target not available for RFE")

INFO:src.feature_engineering.feature_selection:RFE selected 15 features


RFE selected 15 features
Selected features:
  precipitation
  precip_in
  humidity
  cloud_cover
  visibility
  visibility_miles
  pm10
  temp_anomaly
  feels_like_diff
  rain_flag
  rainfall_intensity
  rain_rolling_7d
  cumulative_rainfall
  dew_point
  pollution_score


## 10. Combined Feature Selection

In [10]:
combined_features = selector.combined_selection(
    methods=['variance_threshold', 'correlation', 'rf_importance'],
    min_votes=2
)
print(f"Combined selection selected {len(combined_features)} features")
print("\nSelected features:")
for feat in combined_features:
    print(f"  {feat}")

INFO:src.feature_engineering.feature_selection:Variance threshold selected 54 features
INFO:src.feature_engineering.feature_selection:Correlation selection removed 15 features
INFO:src.feature_engineering.feature_selection:Random Forest importance selected 10 features
INFO:src.feature_engineering.feature_selection:Combined selection selected 42 features


Combined selection selected 42 features

Selected features:
  latitude
  longitude
  temperature
  wind_mph
  wind_degree
  pressure
  precipitation
  humidity
  cloud_cover
  visibility
  uv_index
  gust_mph
  carbon_monoxide
  ozone
  nitrogen_dioxide
  sulphur_dioxide
  pm2_5
  pm10
  air_quality_us-epa-index
  air_quality_gb-defra-index
  moon_illumination
  year
  month
  day
  hour
  day_of_week
  is_weekend
  is_monsoon
  temp_rolling_7d
  temp_trend
  feels_like_diff
  rain_flag
  rain_rolling_7d
  cumulative_rainfall
  gust_difference
  wind_severity
  comfort_index
  dew_point
  fog_indicator
  low_visibility_alert
  pm_ratio
  air_quality_alert


## 11. Feature Selection Report

In [11]:
selector.print_report()

FEATURE SELECTION REPORT

VARIANCE_THRESHOLD:
  Selected: 54 features
  Removed: 7 features

CORRELATION:
  Selected: 46 features
  Removed: 15 features

UNIVARIATE:
  Selected: 20 features
  Top features by score:
    rain_flag: inf
    cloud_cover: 99002.3400
    humidity: 20973.6386
    precipitation: 18599.3304
    rainfall_intensity: 18599.3304

RF_IMPORTANCE:
  Selected: 10 features
  Top features by importance:
    rainfall_intensity: 0.2539
    rain_flag: 0.2490
    precipitation: 0.2330
    precip_in: 0.0903
    cloud_cover: 0.0576

RFE:
  Selected: 15 features

COMBINED:
  Selected: 42 features



## 12. Create Final Dataset with Selected Features

In [12]:
# Use combined features for final dataset
final_features = combined_features

# Add categorical columns if any
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
final_cols = final_features + categorical_cols

# Create final dataset
final_df = df[final_cols].copy()

print(f"Final dataset shape: {final_df.shape}")
print(f"Selected features: {len(final_features)}")
print(f"Categorical features: {len(categorical_cols)}")

Final dataset shape: (119398, 68)
Selected features: 42
Categorical features: 26


## 13. Save Selected Features Dataset

In [13]:
final_df.to_csv('../data/processed/05_selected_features.csv', index=False)
print("\nDataset with selected features saved to ../data/processed/05_selected_features.csv")

# Save feature list for reference
with open('../data/processed/selected_features_list.txt', 'w') as f:
    f.write('\n'.join(final_features))
print("Feature list saved to ../data/processed/selected_features_list.txt")


Dataset with selected features saved to ../data/processed/05_selected_features.csv
Feature list saved to ../data/processed/selected_features_list.txt


## Summary

In this notebook, we:
1. Loaded the EDA-processed data
2. Performed variance threshold selection
3. Removed highly correlated features
4. Applied univariate feature selection (F-test)
5. Used Random Forest feature importance
6. Applied Recursive Feature Elimination
7. Combined multiple selection methods
8. Created final dataset with selected features
9. Saved the selected features for ML modeling

Next: Climate Profiling (Clustering) 